# A preference for A5
This notebook solves the project Euler problem 151 - A preference for A5. The notebook performs the following steps:
- Creates an envelope class
- Performs a tree search of all possible combinations of picking a piece of paper
- Calculates the final result and prints

In [1]:
import pandas as pd
import numpy as np

class Envelope:
    n_combinations = pd.DataFrame({'n_times_opened':range(16), 
                                   'was_a5_n':[0]*16,   'was_not_a5_n':[0]*16, 
                                   'was_a5_p':[0.0]*16, 'was_not_a5_p':[0.0]*16})
    scale_by_paper_size = False
    
    def __init__(self, n_of_each=np.array([1, 1, 1, 1]), p_branch=1, is_copy=False):   # Initialize envelope, n_of_each = [n_A2, n_A3, n_A4, n_A5]
        assert (len(n_of_each) == 4) and all(n_of_each[i] in range(i+2) for i in range(4)), f"n_of_each must be a list of 4 integers no greater than [1, 2, 3, 4], Was {n_of_each}"

        self.n_of_each = n_of_each    # Number of papers of sizes A2, A3, A4 and A5
        self.n_times_opened = 0       # Number of times the envelope is opened
        self.p_branch = 1             # Probability that we are here
        
        # Check if we are copying or initializing a new envelope. If it's new, update class variable
        # If we just want to copy something, don't do this
        if is_copy is False:
            self.n_combinations.loc[0,'was_not_a5_n'] = self.n_combinations.loc[0,'was_not_a5_n'] + 1

    def copy(self):
        out = Envelope(is_copy=True)
        out.n_of_each = self.n_of_each
        out.n_times_opened = self.n_times_opened
        out.p_branch = self.p_branch
        return(out)

    def open_and_get_a5(self, ind):   # Open the envelope, get a piece of paper, cut until it is A5
        assert (isinstance(ind, int)) and (ind >= 0) and (ind <= 3), f"ind must be an integer between 0 and 3, was {ind}"
        
        if (self.n_of_each[ind] == 0):    # If we do not have any paper this size, do nothing
            pass
        else:
            # Update the times the envelope has been opened
            self.n_times_opened = self.n_times_opened + 1 

            n_of_each_new = self.n_of_each.copy()
            if ind == 3:   # We've picked an A5 from the start
                stats_col_names = ['was_a5_n', 'was_a5_p']

            else:          # We've picked something other than A5
                stats_col_names = ['was_not_a5_n', 'was_not_a5_p']
                # Divide whatever we got and add one new to all subsequent sizes. In reality, for A5 we will add two and remove one, so delta=1
                for i in range(ind+1, 4):
                    n_of_each_new[i] = n_of_each_new[i]+1

            # Update stats
            self.n_combinations.loc[self.n_times_opened,stats_col_names[0]] += self.n_of_each[ind]
            self.n_combinations.loc[self.n_times_opened,stats_col_names[1]] += self.p_branch

            # Update the object
            n_of_each_new[ind] = n_of_each_new[ind] - 1 
            self.n_of_each = n_of_each_new

    def __repr__(self):
        return f"{self.__class__.__name__}({self.n_of_each}, {self.p_branch})"

    def __str__(self):
        return('Number of times opened: ' + str(self.n_times_opened) + ', n_of_each: ' + str(self.n_of_each) + ', p_branch: ' + str(self.p_branch))

______________
## Some tests

In [2]:
# e = Envelope()
# print(e)
# display(e.n_combinations[(e.n_combinations.was_a5_n != 0) | (e.n_combinations.was_not_a5_n != 0)])

# # Copying does not update class variable!
# e2 = e.copy()
# print(e2)
# display(e2.n_combinations[(e2.n_combinations.was_a5_n != 0) | (e2.n_combinations.was_not_a5_n != 0)])
# e2

In [3]:
# e = Envelope()

# print(e)
# display(e.n_combinations[(e.n_combinations.was_a5_n != 0) | (e.n_combinations.was_not_a5_n != 0)])

# e.open_and_get_a5(0)
# print(e)
# display(e.n_combinations[(e.n_combinations.was_a5_n != 0) | (e.n_combinations.was_not_a5_n != 0)])

# e.open_and_get_a5(3)
# print(e)
# display(e.n_combinations[(e.n_combinations.was_a5_n != 0) | (e.n_combinations.was_not_a5_n != 0)])

# e2 = Envelope()
# print(e2)
# display(e2.n_combinations[(e2.n_combinations.was_a5_n != 0) | (e2.n_combinations.was_not_a5_n != 0)])

In [4]:
# e = Envelope()
# print(e)
# e.open_and_get_a5(0)
# print(e)
# e.open_and_get_a5(1)
# print(e)
# e.open_and_get_a5(2)
# print(e)
# e.open_and_get_a5(2)
# print(e)
# e.open_and_get_a5(3)
# print(e)
# e.open_and_get_a5(3)
# print(e)
# e.open_and_get_a5(3)
# print(e)
# e.open_and_get_a5(3)
# print(e)

_______________
## Tree

In [5]:
# Go over the tree
env_list = [Envelope()]    # Initialize
# print(env_list)
done = False
while not done:
    # Take the last envelope (removing it from the list)
#    [print(item) for item in env_list]
#    print(' ')
    e = env_list.pop()

    for i in range(4):
        if e.n_of_each[i] != 0:
            e_new = e.copy()
            # Set probability to pick this branch
            # It is a function of probability so far and, sometimes, size of paper
            if e_new.scale_by_paper_size:
                p_new = e_new.p_branch * e_new.n_of_each[i] * 2**(3-i)/sum(e_new.n_of_each*np.array([8, 4, 2, 1]))
            else:
                p_new = e_new.p_branch * e_new.n_of_each[i]/sum(e_new.n_of_each)
            e_new.p_branch = p_new

            # Pick a sheet and add pack to list
            e_new.open_and_get_a5(i)
            env_list.append(e_new)

    # Check if the last envelope element is empty = Envelope([0, 0, 0, 0]). 
    # If it is, remove it
    while (len(env_list) > 0) and (all(env_list[-1].n_of_each == [0, 0, 0, 0])):
        env_list.pop()

    # See if we have anything left in the list - if not, then we're done
    if len(env_list) == 0:
        done = True

In [6]:
# Print answer
display(e.n_combinations)

n_not_a5_p = e.n_combinations[~e.n_combinations.n_times_opened.isin([0, 15])].was_not_a5_p.sum()
n_a5_p = e.n_combinations[~e.n_combinations.n_times_opened.isin([0, 15])].was_a5_p.sum()
n_tot_p = n_not_a5_p + n_a5_p
print('With probability scaling:', n_a5_p, '/', n_tot_p, '=', round(n_a5_p/n_tot_p,6))

,n_times_opened,was_a5_n,was_not_a5_n,was_a5_p,was_not_a5_p
0,0,0,1,0.000000,0.000000
1,1,1,3,0.250000,0.750000
2,2,6,12,0.308333,0.691667
3,3,24,36,0.376171,0.623829
4,4,84,104,0.400194,0.599806
5,5,264,262,0.448289,0.551711
6,6,744,644,0.469613,0.530387
7,7,1960,1368,0.503611,0.496389
8,8,4648,2796,0.497908,0.502092
9,9,10136,4812,0.548249,0.451751


With probability scaling: 7.000000000000076 / 14.000000000000108 = 0.5


This is *not* the correct solution. I tried using `scale_by_paper_size=True` and that is not correct either. 

I see now that they are looking for the expected *number of times* that an A5 paper comes out of the envelope. I'm going to redo the whole thing now.